# Buku Kerja Hari 3 — Object-Oriented Programming (OOP)

Pendamping **Modul 3**. Estimasi 4–6 jam. Prasyarat: Hari 1–2.

**Tujuan:** membuat class dan object, memakai attribute/method, inheritance dengan `super()`, encapsulation, special methods (`__str__`, `__repr__`, `__eq__`, `__len__`), context manager, serta dua mini project berbasis class.

| Bagian | Isi |
|---|---|
| A | Class, object, attribute, method (L1–L3) |
| B | Inheritance & polimorfisme (L4–L5) |
| C | Encapsulation & special methods (L6–L9) |
| D | Jebakan umum (L10) |
| E | Mini Project 1 & 2 (P1, P2) |
| F | Rekap |

> **Cara berpikir OOP:** object = data (attribute) + perilaku (method). Class adalah cetakannya.

> **Notebook SOLUSI** — semua latihan sudah terjawab dan dieksekusi; gunakan untuk membandingkan dengan jawaban Anda.

In [1]:
# @title Jalankan sel ini lebih dulu (alat bantu pemeriksa jawaban)
from contextlib import contextmanager
from unittest.mock import patch

STRICT = True
_hasil = {}

@contextmanager
def periksa(kode):
    """Dipakai di sel 'Periksa jawaban': gagal -> pesan petunjuk; lolos -> tanda centang."""
    try:
        yield
        _hasil[kode] = True
        print(f"✅ {kode} benar!")
    except AssertionError as e:
        _hasil[kode] = False
        print(f"❌ {kode} belum tepat: {e}")
        if STRICT: raise
    except Exception as e:
        _hasil[kode] = False
        print(f"❌ {kode} belum bisa dijalankan: {type(e).__name__}: {e}")
        if STRICT: raise

@contextmanager
def masukan(*jawaban):
    """Simulasi input(): with masukan('Ani', '80'): ... -> input() mengembalikan 'Ani' lalu '80'."""
    with patch("builtins.input", side_effect=list(jawaban)):
        yield

def ringkasan(semua):
    """Cetak rekap latihan: ✅ benar, ❌ salah, ⏳ belum dicoba."""
    benar = 0
    for kode in semua:
        s = _hasil.get(kode)
        tanda = "⏳ belum dicoba" if s is None else ("✅ benar" if s else "❌ perlu diperbaiki")
        benar += bool(s)
        print(f"{kode:<6} {tanda}")
    print(f"\nSkor: {benar} dari {len(semua)} latihan benar")

from pathlib import Path
WORK = Path("latihan_hari3")
WORK.mkdir(exist_ok=True)


## A. Class, Object, Attribute, Method

**Ringkas:** `class Nama:` mendefinisikan cetakan; `__init__(self, ...)` dipanggil saat object dibuat dan mengisi *instance attribute* (`self.x = ...`). Method adalah function di dalam class dengan `self` sebagai parameter pertama. *Class attribute* dipakai bersama semua object.

In [2]:
class Mahasiswa:
    jumlah = 0                      # class attribute (milik class)

    def __init__(self, nama):
        self.nama = nama            # instance attribute (milik object)
        Mahasiswa.jumlah += 1

    def sapa(self):
        return f"Halo, saya {self.nama}"

a = Mahasiswa("Ani"); b = Mahasiswa("Budi")
print(a.sapa(), "|", b.nama, "|", Mahasiswa.jumlah)

Halo, saya Ani | Budi | 2


### Latihan L1 — Class Mahasiswa

Buat class `Mahasiswa` dengan `__init__(self, nama, nim, prodi, ipk=0.0)`, method `info()` yang mengembalikan `"<nama> (<nim>) — <prodi>, IPK <ipk>"`, dan `update_ipk(baru)` yang mengubah `ipk` bila 0–4 lalu mengembalikan `True`; selain itu `False` (ipk tidak berubah).

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
CLASS Mahasiswa
    __init__(nama, nim, prodi, ipk=0.0): simpan ke self.*
    info(): f-string sesuai format
    update_ipk(baru): JIKA 0..4 → ubah self.ipk, True; SELAIN → False
```

</details>

In [3]:
class Mahasiswa:
    def __init__(self, nama, nim, prodi, ipk=0.0):
        self.nama = nama
        self.nim = nim
        self.prodi = prodi
        self.ipk = ipk

    def info(self):
        return f"{self.nama} ({self.nim}) — {self.prodi}, IPK {self.ipk}"

    def update_ipk(self, baru):
        if 0 <= baru <= 4:
            self.ipk = baru
            return True
        return False

In [4]:
# Periksa jawaban
with periksa('L1'):
    m = Mahasiswa("Ani", "2201", "Informatika", 3.75)
    assert m.info() == "Ani (2201) — Informatika, IPK 3.75", f"info: {m.info()!r}"
    m2 = Mahasiswa("Budi", "2202", "Sistem Informasi")
    assert m2.ipk == 0.0, "ipk default 0.0"
    assert m2.update_ipk(3.4) is True and m2.ipk == 3.4
    assert m2.update_ipk(5) is False and m2.ipk == 3.4, "IPK tidak valid tidak boleh mengubah data"

✅ L1 benar!


### Latihan L2 — Class attribute sebagai penghitung

Buat class `Produk` dengan class attribute `jumlah = 0` yang bertambah setiap object dibuat (di `__init__`), dan instance attribute `nama`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
jumlah ← 0 di level class
di __init__: simpan nama, lalu Produk.jumlah += 1
```

</details>

In [5]:
class Produk:
    jumlah = 0

    def __init__(self, nama):
        self.nama = nama
        Produk.jumlah += 1

In [6]:
# Periksa jawaban
with periksa('L2'):
    Produk.jumlah = 0
    a, b, c = Produk("Pensil"), Produk("Buku"), Produk("Penghapus")
    assert Produk.jumlah == 3, f"jumlah: {Produk.jumlah}"
    assert (a.nama, b.nama) == ("Pensil", "Buku"), "nama adalah instance attribute (berbeda per object)"

✅ L2 benar!


### Latihan L3 — RekeningBank

Buat `RekeningBank(pemilik, saldo=0)` dengan method `setor(jumlah)` (hanya jumlah > 0, kembalikan `True/False`) dan `tarik(jumlah)` (hanya `0 < jumlah <= saldo`, kembalikan `True/False`). Atribut `saldo` bisa dibaca langsung.

In [7]:
class RekeningBank:
    def __init__(self, pemilik, saldo=0):
        self.pemilik = pemilik
        self.saldo = saldo

    def setor(self, jumlah):
        if jumlah > 0:
            self.saldo += jumlah
            return True
        return False

    def tarik(self, jumlah):
        if 0 < jumlah <= self.saldo:
            self.saldo -= jumlah
            return True
        return False

In [8]:
# Periksa jawaban
with periksa('L3'):
    r = RekeningBank("Ani", 500000)
    assert r.setor(200000) is True and r.saldo == 700000
    assert r.tarik(100000) is True and r.saldo == 600000
    assert r.tarik(2000000) is False and r.saldo == 600000, "saldo tidak cukup → gagal, saldo tetap"
    assert r.setor(-5) is False and r.saldo == 600000

✅ L3 benar!


## B. Inheritance dan Polimorfisme

**Ringkas:** `class Anak(Induk):` mewarisi attribute/method induk. Di `__init__` anak panggil `super().__init__(...)` agar attribute induk terisi. *Override* = menulis ulang method dengan nama sama; `super().method()` memanggil versi induk.

In [9]:
class Pegawai:
    def __init__(self, nama, gaji):
        self.nama, self.gaji = nama, gaji
    def info(self):
        return f"{self.nama}: {self.gaji}"

class Dosen(Pegawai):
    def __init__(self, nama, gaji, bidang):
        super().__init__(nama, gaji)
        self.bidang = bidang
    def info(self):
        return super().info() + f" | {self.bidang}"

d = Dosen("Dr. Sari", 12000000, "ML")
print(d.info(), isinstance(d, Pegawai), issubclass(Dosen, Pegawai))

Dr. Sari: 12000000 | ML True True


### Latihan L4 — Kendaraan, Mobil, Motor

Buat `Kendaraan(merek, tahun)` dengan `info()` → `"<merek> (<tahun>)"`. Turunkan `Mobil(merek, tahun, jumlah_pintu)` dengan `info()` → `"[Mobil] <info induk> — <n> pintu"` dan `Motor(merek, tahun, jenis)` dengan `info()` → `"[Motor] <info induk> — <jenis>"`. Gunakan `super()`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
Mobil.__init__: super().__init__(merek, tahun); simpan jumlah_pintu
Mobil.info: f"[Mobil] {super().info()} — {pintu} pintu"
Motor serupa dengan 'jenis'
```

</details>

In [10]:
class Kendaraan:
    def __init__(self, merek, tahun):
        self.merek = merek
        self.tahun = tahun

    def info(self):
        return f"{self.merek} ({self.tahun})"

class Mobil(Kendaraan):
    def __init__(self, merek, tahun, jumlah_pintu):
        super().__init__(merek, tahun)
        self.jumlah_pintu = jumlah_pintu

    def info(self):
        return f"[Mobil] {super().info()} — {self.jumlah_pintu} pintu"

class Motor(Kendaraan):
    def __init__(self, merek, tahun, jenis):
        super().__init__(merek, tahun)
        self.jenis = jenis

    def info(self):
        return f"[Motor] {super().info()} — {self.jenis}"

In [11]:
# Periksa jawaban
with periksa('L4'):
    assert Kendaraan("Honda", 2020).info() == "Honda (2020)"
    assert Mobil("Toyota", 2022, 4).info() == "[Mobil] Toyota (2022) — 4 pintu"
    assert Motor("Honda", 2023, "Matic").info() == "[Motor] Honda (2023) — Matic"
    assert issubclass(Mobil, Kendaraan) and issubclass(Motor, Kendaraan)

✅ L4 benar!


### Latihan L5 — Polimorfisme: Pegawai dan Manager

Buat `Pegawai(nama, gaji_pokok)` dengan `gaji()` → `gaji_pokok`, dan `Manager(nama, gaji_pokok, tunjangan)` yang **meng-override** `gaji()` → `gaji_pokok + tunjangan`. Lalu isi list `tim` dengan `Pegawai("Rina", 7_000_000)` dan `Manager("Budi", 9_000_000, 3_000_000)`, dan hitung `total_gaji` dengan **satu loop** yang memanggil `gaji()` pada setiap object.

In [12]:
class Pegawai:
    def __init__(self, nama, gaji_pokok):
        self.nama = nama
        self.gaji_pokok = gaji_pokok

    def gaji(self):
        return self.gaji_pokok

class Manager(Pegawai):
    def __init__(self, nama, gaji_pokok, tunjangan):
        super().__init__(nama, gaji_pokok)
        self.tunjangan = tunjangan

    def gaji(self):
        return self.gaji_pokok + self.tunjangan

tim = [Pegawai("Rina", 7_000_000), Manager("Budi", 9_000_000, 3_000_000)]
total_gaji = 0
for p in tim:
    total_gaji += p.gaji()

In [13]:
# Periksa jawaban
with periksa('L5'):
    assert Pegawai("A", 100).gaji() == 100
    assert Manager("B", 100, 50).gaji() == 150
    assert [type(p).__name__ for p in tim] == ["Pegawai", "Manager"], f"isi tim: {tim}"
    assert total_gaji == 19_000_000, f"total_gaji: {total_gaji}"

✅ L5 benar!


## C. Encapsulation, Special Methods, Context Manager

**Ringkas:** *encapsulation* = data diubah lewat method resmi (validasi). Awalan `_x` = konvensi "internal". Special methods membuat object berperilaku seperti tipe bawaan: `__str__` (untuk `print`), `__repr__` (representasi developer), `__len__`, `__eq__`. Context manager (`with`) memakai `__enter__`/`__exit__`.

In [14]:
class Buku:
    def __init__(self, judul, halaman):
        self.judul, self.halaman = judul, halaman
    def __str__(self):
        return f"Buku: {self.judul}"
    def __len__(self):
        return self.halaman
    def __eq__(self, other):
        return self.judul == other.judul

b1, b2 = Buku("Python", 250), Buku("Python", 300)
print(b1, len(b1), b1 == b2)

Buku: Python 250 True


### Latihan L6 — Encapsulation — Rekening aman

Buat `Rekening(pemilik, saldo=0)` yang menyimpan saldo di `self._saldo`, dengan `get_saldo()`, `setor(jumlah)` (hanya > 0) dan `tarik(jumlah)` (hanya `0 < jumlah <= saldo`). Kedua method mengembalikan `True/False`. **Jangan** akses `_saldo` dari luar class.

In [15]:
class Rekening:
    def __init__(self, pemilik, saldo=0):
        self.pemilik = pemilik
        self._saldo = saldo

    def get_saldo(self):
        return self._saldo

    def setor(self, jumlah):
        if jumlah > 0:
            self._saldo += jumlah
            return True
        return False

    def tarik(self, jumlah):
        if 0 < jumlah <= self._saldo:
            self._saldo -= jumlah
            return True
        return False

In [16]:
# Periksa jawaban
with periksa('L6'):
    r = Rekening("Ani", 1000)
    assert r.setor(500) is True and r.get_saldo() == 1500
    assert r.tarik(2000) is False and r.get_saldo() == 1500
    assert r.tarik(500) is True and r.get_saldo() == 1000
    assert r.setor(0) is False
    assert hasattr(r, "_saldo") and not hasattr(r, "saldo"), "simpan di _saldo, bukan saldo"

✅ L6 benar!


### Latihan L7 — Special methods pada Buku

Buat `Buku(judul, penulis, halaman)` dengan: `__str__` → `"'<judul>' oleh <penulis>"`; `__repr__` → `"Buku('<judul>', '<penulis>', <halaman>)"`; `__len__` → `halaman`; `__eq__` → sama bila `judul` **dan** `penulis` sama.

In [17]:
class Buku:
    def __init__(self, judul, penulis, halaman):
        self.judul = judul
        self.penulis = penulis
        self.halaman = halaman

    def __str__(self):
        return f"'{self.judul}' oleh {self.penulis}"

    def __repr__(self):
        return f"Buku('{self.judul}', '{self.penulis}', {self.halaman})"

    def __len__(self):
        return self.halaman

    def __eq__(self, other):
        return self.judul == other.judul and self.penulis == other.penulis

In [18]:
# Periksa jawaban
with periksa('L7'):
    b1 = Buku("Python Dasar", "Ani", 250)
    b2 = Buku("Python Dasar", "Ani", 300)
    b3 = Buku("Python Lanjut", "Ani", 250)
    assert str(b1) == "'Python Dasar' oleh Ani", f"str: {str(b1)!r}"
    assert repr(b1) == "Buku('Python Dasar', 'Ani', 250)", f"repr: {b1!r}"
    assert len(b1) == 250
    assert b1 == b2 and b1 != b3

✅ L7 benar!


### Latihan L8 — Titik dan jarak

Buat `Titik(x, y)` dengan `jarak_ke(lain)` (jarak Euclid; boleh pakai `math.hypot`) dan `__str__` → `"(x, y)"`.

In [19]:
import math

class Titik:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def jarak_ke(self, lain):
        return math.hypot(self.x - lain.x, self.y - lain.y)

    def __str__(self):
        return f"({self.x}, {self.y})"

In [20]:
# Periksa jawaban
with periksa('L8'):
    a, b = Titik(0, 0), Titik(3, 4)
    assert a.jarak_ke(b) == 5.0
    assert str(Titik(1, 2)) == "(1, 2)"

✅ L8 benar!


### Latihan L9 — Context manager FileCatatan

Buat class `FileCatatan(path, mode="w")` sebagai context manager: `__enter__` membuka file (encoding utf-8) dan mengembalikannya, `__exit__` menutup file **dan tidak menelan exception** (kembalikan `False`).

In [21]:
class FileCatatan:
    def __init__(self, path, mode="w"):
        self.path = path
        self.mode = mode
        self.file = None

    def __enter__(self):
        self.file = open(self.path, self.mode, encoding="utf-8")
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()
        return False

In [22]:
# Periksa jawaban
with periksa('L9'):
    p = WORK / "catatan.txt"
    with FileCatatan(p) as f:
        f.write("Belajar OOP\n")
    assert f.closed, "file harus tertutup setelah blok with"
    assert p.read_text(encoding="utf-8") == "Belajar OOP\n"
    try:
        with FileCatatan(p, "a") as f2:
            raise ValueError("uji")
    except ValueError:
        assert f2.closed, "file harus tetap ditutup walau terjadi error"
    else:
        raise AssertionError("exception tidak boleh ditelan (return False di __exit__)")

✅ L9 benar!


## D. Jebakan Umum

Dua jebakan yang paling sering: (1) lupa `self`; (2) *mutable default argument* — list/dict sebagai nilai default dibuat **sekali** dan dipakai bersama semua object.

In [23]:
class Salah:
    def __init__(self, item=[]):
        self.item = item

x, y = Salah(), Salah()
x.item.append("A")
print(y.item, x.item is y.item)     # y ikut berubah!

['A'] True


### Latihan L10 — Perbaiki mutable default

Perbaiki class `Kelas` agar setiap object punya list `mahasiswa` sendiri: pakai default `None` lalu buat list baru di `__init__`.

<details><summary>💡 Petunjuk (pseudocode)</summary>

```text
def __init__(self, nama, mahasiswa=None):
    self.mahasiswa ← mahasiswa JIKA bukan None, SELAIN ITU list kosong baru
```

</details>

In [24]:
class Kelas:
    def __init__(self, nama, mahasiswa=None):
        self.nama = nama
        self.mahasiswa = mahasiswa if mahasiswa is not None else []

In [25]:
# Periksa jawaban
with periksa('L10'):
    a, b = Kelas("A"), Kelas("B")
    a.mahasiswa.append("Ani")
    assert b.mahasiswa == [], f"b.mahasiswa harus kosong, isinya {b.mahasiswa}"
    assert a.mahasiswa is not b.mahasiswa
    c = Kelas("C", ["Budi"])
    assert c.mahasiswa == ["Budi"]

✅ L10 benar!


## E. Mini Project

### Latihan P1 — Mini Project 1 — Manajemen Tugas Kuliah

Buat dua class:

- `Tugas(matkul, deskripsi, deadline)` dengan `selesai = False`, `tandai_selesai()`, dan `__str__` → `"[✓] matkul: deskripsi (deadline)"` bila selesai, `"[○] ..."` bila belum.
- `ManajerTugas` dengan `daftar` (list), `tambah(tugas)`, `tampilkan()` (kembalikan **list string** bernomor mulai 1: `"1. [○] ..."`), `tandai(index)` (index mulai 1; kembalikan `True`/`False` bila index tidak valid), dan `belum_selesai()` (list `Tugas` yang belum selesai).

In [26]:
class Tugas:
    def __init__(self, matkul, deskripsi, deadline):
        self.matkul = matkul
        self.deskripsi = deskripsi
        self.deadline = deadline
        self.selesai = False

    def tandai_selesai(self):
        self.selesai = True

    def __str__(self):
        st = "✓" if self.selesai else "○"
        return f"[{st}] {self.matkul}: {self.deskripsi} ({self.deadline})"

class ManajerTugas:
    def __init__(self):
        self.daftar = []

    def tambah(self, tugas):
        self.daftar.append(tugas)

    def tampilkan(self):
        return [f"{i}. {t}" for i, t in enumerate(self.daftar, 1)]

    def tandai(self, index):
        if 1 <= index <= len(self.daftar):
            self.daftar[index - 1].tandai_selesai()
            return True
        return False

    def belum_selesai(self):
        return [t for t in self.daftar if not t.selesai]

In [27]:
# Periksa jawaban
with periksa('P1'):
    m = ManajerTugas()
    m.tambah(Tugas("Python", "Laporan", "2026-10-05"))
    m.tambah(Tugas("Basis Data", "ERD", "2026-10-07"))
    assert m.tampilkan() == ["1. [○] Python: Laporan (2026-10-05)", "2. [○] Basis Data: ERD (2026-10-07)"], m.tampilkan()
    assert m.tandai(2) is True and m.tandai(9) is False
    assert m.tampilkan()[1] == "2. [✓] Basis Data: ERD (2026-10-07)"
    assert [t.matkul for t in m.belum_selesai()] == ["Python"]

✅ P1 benar!


### Latihan P2 — Mini Project 2 — Perpustakaan dengan Inheritance

Buat hierarki:

- `ItemPerpustakaan(judul)` dengan `tersedia = True`, `pinjam()` (jika tersedia → set `False`, kembalikan `True`; selain itu `False`), `kembalikan()`.
- `Buku(judul, penulis, halaman)` → `__str__`: `"Buku: <judul> — <penulis> [tersedia/dipinjam]"`.
- `Jurnal(judul, volume, tahun)` → `__str__`: `"Jurnal: <judul> Vol.<volume> (<tahun>) [tersedia/dipinjam]"`.
- `Perpustakaan` dengan `koleksi`, `tambah(item)`, `cari(keyword)` (judul mengandung kata, tak peka huruf besar/kecil), `pinjam(judul)` (kembalikan `True/False`, judul tak peka huruf besar/kecil), dan `daftar()` → list `str(item)` seluruh koleksi (**polimorfisme**: satu loop untuk semua jenis).

In [28]:
class ItemPerpustakaan:
    def __init__(self, judul):
        self.judul = judul
        self.tersedia = True

    def pinjam(self):
        if self.tersedia:
            self.tersedia = False
            return True
        return False

    def kembalikan(self):
        self.tersedia = True

    def _status(self):
        return "tersedia" if self.tersedia else "dipinjam"

class Buku(ItemPerpustakaan):
    def __init__(self, judul, penulis, halaman):
        super().__init__(judul)
        self.penulis = penulis
        self.halaman = halaman

    def __str__(self):
        return f"Buku: {self.judul} — {self.penulis} [{self._status()}]"

class Jurnal(ItemPerpustakaan):
    def __init__(self, judul, volume, tahun):
        super().__init__(judul)
        self.volume = volume
        self.tahun = tahun

    def __str__(self):
        return f"Jurnal: {self.judul} Vol.{self.volume} ({self.tahun}) [{self._status()}]"

class Perpustakaan:
    def __init__(self):
        self.koleksi = []

    def tambah(self, item):
        self.koleksi.append(item)

    def cari(self, keyword):
        k = keyword.lower()
        return [i for i in self.koleksi if k in i.judul.lower()]

    def pinjam(self, judul):
        for i in self.koleksi:
            if i.judul.lower() == judul.lower():
                return i.pinjam()
        return False

    def daftar(self):
        return [str(i) for i in self.koleksi]

In [29]:
# Periksa jawaban
with periksa('P2'):
    p = Perpustakaan()
    p.tambah(Buku("Python Dasar", "Ani", 250))
    p.tambah(Jurnal("Jurnal Informatika", 12, 2025))
    assert p.daftar() == ["Buku: Python Dasar — Ani [tersedia]", "Jurnal: Jurnal Informatika Vol.12 (2025) [tersedia]"], p.daftar()
    assert p.pinjam("python dasar") is True
    assert p.pinjam("Python Dasar") is False, "sudah dipinjam"
    assert p.pinjam("Tidak Ada") is False
    assert p.daftar()[0] == "Buku: Python Dasar — Ani [dipinjam]"
    assert [i.judul for i in p.cari("INFORMATIKA")] == ["Jurnal Informatika"]
    assert isinstance(p.koleksi[0], ItemPerpustakaan) and isinstance(p.koleksi[1], ItemPerpustakaan)
    p.koleksi[0].kembalikan()
    assert p.koleksi[0].tersedia is True

✅ P2 benar!


## F. Rekap

1. Dengan kata sendiri: apa beda *class* dan *object*? Beri contoh dari kehidupan kampus.
2. Kapan inheritance cocok, dan kapan lebih baik object memegang object lain (komposisi)?
3. Satu error OOP yang Anda temui hari ini (mis. `TypeError: __init__() missing ...`, `AttributeError`) dan perbaikannya:

> *Jawaban Anda:* …

**Checklist sebelum Hari 4**

- [ ] Semua latihan L1–L10 berstatus ✅
- [ ] Mini Project 1 (P1) dan 2 (P2) lulus pemeriksaan
- [ ] Saya paham fungsi `self` dan `super()`
- [ ] Saya bisa menulis `__str__`, `__repr__`, `__eq__`
- [ ] Saya tahu mengapa tidak boleh memakai list sebagai default argument

## Rekap hasil latihan

Jalankan sel di bawah setelah mengerjakan latihan.

In [30]:
ringkasan(['L1', 'L2', 'L3', 'L4', 'L5', 'L6', 'L7', 'L8', 'L9', 'L10', 'P1', 'P2'])

L1     ✅ benar
L2     ✅ benar
L3     ✅ benar
L4     ✅ benar
L5     ✅ benar
L6     ✅ benar
L7     ✅ benar
L8     ✅ benar
L9     ✅ benar
L10    ✅ benar
P1     ✅ benar
P2     ✅ benar

Skor: 12 dari 12 latihan benar
